## Stocks Market Data Analysis nased on Global Cues

In [36]:
from dotenv import load_dotenv

In [37]:
from langchain_community.document_loaders import WebBaseLoader

urls = ['https://economictimes.indiatimes.com/markets/stocks/news',
        'https://www.livemint.com/latest-news',
        'https://www.livemint.com/latest-news/page-2',
        'https://www.livemint.com/latest-news/page-3',
        'https://www.moneycontrol.com/']

In [38]:
loader = WebBaseLoader(web_paths=urls)

In [39]:
from bs4 import BeautifulSoup
docs = []
for doc in loader.load():
    docs.append(doc)


In [40]:
docs

[Document(metadata={'source': 'https://economictimes.indiatimes.com/markets/stocks/news', 'title': 'Stocks in News Today - Latest News on Stocks, Stock in News | The Economic Times', 'description': 'Stocks in News - Find the latest Stocks in News on The Economic Times. Get Stocks Analysis, News on Stocks and more.', 'language': 'en'}, page_content='Stocks in News Today - Latest News on Stocks, Stock in News | The Economic TimesBenchmarks Nifty23,897.95-275.1FEATURED FUNDS★★★★★Motilal Oswal Midcap Fund Direct-Growth5Y Return24.97 %\n                Invest NowEnter search text:English EditionEnglish Editionहिन्दीગુજરાતીमराठीবাংলাಕನ್ನಡമലയാളംதமிழ்తెలుగు | 26 April, 2026, 03:36 PM IST | Today\'s ePaper\n            \t\t\t        My Watchlist\n                            SubscribeSign InHomeETPrimeMarketsMarket DataMasterclassIPONewsIndustrySMEPoliticsWealthMFTechAICareersOpinionNRIPanacheMore MenuStocksNewsLive BlogStock Live BlogEarningsPodcastMarket ClassroomDons of Dalal StreetRecosStock

In [41]:
def format_docs(docs):
    return "\n\n".join([x.page_content for x in docs])

In [42]:
context = format_docs(docs)

In [43]:
# print(context)
# context

import re

def text_clean(text):
    text = re.sub(r'\n\n+', '\n\n', text)
    text = re.sub(r'\t+', '\t', text)
    text = re.sub(r'\s+', ' ', text)
    return text

In [44]:
context = text_clean(context)

In [45]:
print(context)

Stocks in News Today - Latest News on Stocks, Stock in News | The Economic TimesBenchmarks Nifty23,897.95-275.1FEATURED FUNDS★★★★★Motilal Oswal Midcap Fund Direct-Growth5Y Return24.97 % Invest NowEnter search text:English EditionEnglish Editionहिन्दीગુજરાતીमराठीবাংলাಕನ್ನಡമലയാളംதமிழ்తెలుగు | 26 April, 2026, 03:36 PM IST | Today's ePaper My Watchlist SubscribeSign InHomeETPrimeMarketsMarket DataMasterclassIPONewsIndustrySMEPoliticsWealthMFTechAICareersOpinionNRIPanacheMore MenuStocksNewsLive BlogStock Live BlogEarningsPodcastMarket ClassroomDons of Dalal StreetRecosStock Reports PlusNewMy ScreenerCandlestick ScreenerStock ScreenerStock WatchMarket CalendarStock Price QuotesOptionsIPOs/FPOsExpert ViewsInvestment IdeasCommoditiesViewsNewsOthersMentha OilPrecious MetalsGold MGoldSilverGold PetalSilver MicroSilver MGold GuineaSpicesCardamomOil & EnergyNatural GasCrude OilCrude Oil MiniBase MetalsAluminiumZinc MiniLead MiniCopperZincNickelAluminium MiniLeadPlantationKapasCottonForexForex News

Stock Market Data Processing with LLM

In [47]:
from scripts import llmGroq

response = llmGroq.ask_llm(context[:500], "what is today's news?")

### Castrophic Forgetting
where llm just emphasis on starting and ending of your data (incase of large data)
Solution: you break your data into smaller chunks and then combine them

In [50]:
from scripts import llmGroq

response = llmGroq.ask_llm(context[:5000], "what is today's news?")

In [51]:
print(response)

Nifty ended lower on Friday, dragged mainly by IT stocks, while pharma, healthcare and energy stocks also remained under pressure. The past three sessions have been highly volatile, with the index witnessing a sharp decline amid renewed US-Iran tensions.


In [52]:
def chunk_text(text, chunk_size, overlap=100):
    chunks = []
    for i in range(0,len(text), chunk_size-overlap):
        chunks.append(text[i:i + chunk_size])
    return chunks



In [53]:
chunks = chunk_text(context, 5000)

In [57]:
question = "Extract stock market news from the given text."

chunk_summary = []
for chunk in chunks:
    response = llmGroq.ask_llm(chunk, question)
    chunk_summary.append(response)

KeyboardInterrupt: 

In [59]:
for chunk in chunk_summary:
    print(chunk)
    print("\n\n")
    break

Nifty ended lower on Friday, dragged mainly by IT stocks, while pharma, healthcare and energy stocks also remained under pressure. The past three sessions have been highly volatile, with the index witnessing a sharp decline amid renewed US-Iran tensions.

Decoding the charts, Rupak De, Senior Technical Analyst at LKP Securities, said the index faced resistance near its 100-day EMA on the daily chart, which capped the rally and triggered fresh selling, dragging it below the 24,000 mark. The broader setup now appears bearish, with Nifty likely to drift towards 23,500. However, he added that 24,200 remains an immediate resistance level, and a move above it could help improve market sentiment.

The combined market valuation of seven of the top-10 most-valued firms eroded by Rs 2 lakh crore last week, with Tata Consultancy Services and Reliance Industries emerging as the biggest laggards, in-tandem with a bearish trend in equities.

The Indian stock market has seen sharp downturns over the 

In [60]:
summary = "\n\n".join(chunk_summary)

In [ ]:
# print(summary)

In [62]:
question = """Write a detailed market news report in markdown format. Think carefully then write the report."""
response = llmGroq.ask_llm(summary, question)

In [63]:
import os

os.makedirs("data", exist_ok=True)

with open("data/report.md", "w", encoding="utf-8") as f:
    f.write(response)

In [64]:
with open("data/summary.md", "w", encoding="utf-8") as f:
    f.write(summary)